# Bringing Graph Analytics to Snowflake with Neo4j

In this notebook we'll use the Neo4j Graph Analytics Native App to analyze a manufacturing plant dataset entirely inside Snowflake. By the end we'll have:

- Loaded a manufacturing plant dataset into Snowflake
- Run six graph algorithms using the Neo4j Graph Analytics Native App
- Simulated a machine failure and analyzed its impact on the network
- Detected operational communities within the plant
- Interpreted the results to identify connected, critical and similar machines

**Before you start**, ensure the Neo4j Graph Analytics app is installed in your Snowflake account from the Snowflake Marketplace.

**How to use this notebook:** Run each cell in order using the ▶ button or `Shift+Enter`. Do not skip cells - later sections depend on earlier ones.

### What we'll cover

| Section | Topic |
|---|---|
| 0 | Setup |
| 1 | Thinking in Graphs |
| 2 | Your First Graph Projection |
| 3 | Connectivity Analysis |
| 4 | Criticality Analysis |
| 5 | Structural Similarity |
| 6 | Failure Simulation |
| 7 | Community Detection |
| 8 | Risk Summary |


## 0. Setup

This section creates the database, loads the demo data and configures the permissions the Neo4j Graph Analytics app needs to read from and write to tables.

### The scenario

We're working with a manufacturing plant that has 20 machines - Cutters, Welders, Presses, Assemblers and Painters. Each machine has a **risk level** (low, medium or high). Machines are connected by directed material flow relationships, each with a **throughput rate**.

This is exactly the kind of data that already exists in Snowflake for real operational systems. We'll use graph analytics to surface insights that SQL alone cannot easily produce.


In [ ]:
# --------------------------------------------------------------
# Database and schema
# --------------------------------------------------------------
from snowflake.snowpark.context import get_active_session

session = get_active_session()

session.sql("USE ROLE accountadmin").collect()
session.sql("DROP DATABASE IF EXISTS ga_demo").collect()
session.sql("CREATE DATABASE IF NOT EXISTS ga_demo").collect()
session.sql("USE SCHEMA ga_demo.public").collect()

print("Database ga_demo created.")

In [ ]:
# --------------------------------------------------------------
# Create tables
# --------------------------------------------------------------
session.sql("""
    CREATE OR REPLACE TABLE ga_demo.public.nodes (
        machine_id     NUMBER(38, 0),
        machine_type   VARCHAR,
        current_status VARCHAR,
        risk_level     VARCHAR
    )
""").collect()

session.sql("""
    CREATE OR REPLACE TABLE ga_demo.public.rels (
        src_machine_id NUMBER(38, 0),
        dst_machine_id NUMBER(38, 0),
        throughput_rate NUMBER(38, 0)
    )
""").collect()

print("Tables created.")

In [ ]:
# --------------------------------------------------------------
# Load node data - 20 machines
# --------------------------------------------------------------
session.sql("""
    INSERT INTO ga_demo.public.nodes VALUES
    (1,  'Cutter',    'active', 'low'),
    (2,  'Welder',    'active', 'low'),
    (3,  'Press',     'active', 'medium'),
    (4,  'Assembler', 'active', 'medium'),
    (5,  'Paint',     'active', 'low'),
    (6,  'Cutter',    'active', 'low'),
    (7,  'Welder',    'active', 'medium'),
    (8,  'Press',     'active', 'medium'),
    (9,  'Assembler', 'active', 'low'),
    (10, 'Paint',     'active', 'low'),
    (11, 'Cutter',    'active', 'medium'),
    (12, 'Welder',    'active', 'high'),
    (13, 'Press',     'active', 'medium'),
    (14, 'Assembler', 'active', 'high'),
    (15, 'Paint',     'active', 'medium'),
    (16, 'Cutter',    'active', 'low'),
    (17, 'Welder',    'active', 'medium'),
    (18, 'Press',     'active', 'low'),
    (19, 'Assembler', 'active', 'medium'),
    (20, 'Assembler', 'active', 'high')
""").collect()

print("Node data loaded.")

In [ ]:
# --------------------------------------------------------------
# Load relationship data - directed material flow
# --------------------------------------------------------------
session.sql("""
    INSERT INTO ga_demo.public.rels VALUES
    (1, 2, 50),   (2, 3, 50),   (3, 4, 50),   (4, 5, 50),
    (5, 6, 50),   (6, 7, 50),   (7, 8, 50),   (8, 9, 50),
    (9, 10, 50),  (1, 20, 200), (2, 20, 180), (3, 20, 160),
    (4, 20, 140), (11, 12, 20), (12, 13, 20), (13, 14, 20),
    (14, 15, 20), (15, 16, 20), (16, 17, 20), (17, 18, 20),
    (18, 19, 20), (19, 20, 20), (3, 11, 120), (10, 19, 19)
""").collect()

print("Relationship data loaded.")

In [ ]:
# --------------------------------------------------------------
# Permissions - grant the Native App access to ga_demo
# --------------------------------------------------------------
session.sql("USE ROLE accountadmin").collect()
session.sql("USE WAREHOUSE neo4j_graph_analytics_app_warehouse").collect()

# Account-level role for users
session.sql("CREATE ROLE IF NOT EXISTS gds_role").collect()
session.sql("GRANT ALL PRIVILEGES ON ALL VIEWS IN SCHEMA ga_demo.public TO ROLE gds_role").collect()
session.sql("GRANT CREATE VIEW ON SCHEMA ga_demo.public TO ROLE gds_role").collect()

# Grant the app access to the database and schema
session.sql("GRANT USAGE ON DATABASE ga_demo TO APPLICATION neo4j_graph_analytics").collect()
session.sql("GRANT USAGE ON SCHEMA ga_demo.public TO APPLICATION neo4j_graph_analytics").collect()

# Database role for fine-grained table access
session.sql("CREATE DATABASE ROLE IF NOT EXISTS gds_db_role").collect()
session.sql("GRANT ALL PRIVILEGES ON FUTURE TABLES IN SCHEMA ga_demo.public TO DATABASE ROLE gds_db_role").collect()
session.sql("GRANT ALL PRIVILEGES ON ALL TABLES IN SCHEMA ga_demo.public TO DATABASE ROLE gds_db_role").collect()
session.sql("GRANT ALL PRIVILEGES ON FUTURE VIEWS IN SCHEMA ga_demo.public TO DATABASE ROLE gds_db_role").collect()
session.sql("GRANT ALL PRIVILEGES ON ALL VIEWS IN SCHEMA ga_demo.public TO DATABASE ROLE gds_db_role").collect()
session.sql("GRANT CREATE TABLE ON SCHEMA ga_demo.public TO DATABASE ROLE gds_db_role").collect()

# Grant the database role to the app and the account role
session.sql("GRANT DATABASE ROLE gds_db_role TO APPLICATION neo4j_graph_analytics").collect()
session.sql("GRANT DATABASE ROLE gds_db_role TO ROLE gds_role").collect()

# Additional grants to the account role
session.sql("GRANT USAGE ON DATABASE ga_demo TO ROLE gds_role").collect()
session.sql("GRANT USAGE ON SCHEMA ga_demo.public TO ROLE gds_role").collect()
session.sql("GRANT SELECT, INSERT, UPDATE, DELETE ON ALL TABLES IN SCHEMA ga_demo.public TO ROLE gds_role").collect()
session.sql("GRANT CREATE TABLE ON SCHEMA ga_demo.public TO ROLE gds_role").collect()
session.sql("GRANT SELECT, INSERT, UPDATE, DELETE ON FUTURE TABLES IN SCHEMA ga_demo.public TO ROLE gds_role").collect()

print("Permissions configured.")

In [ ]:
# --------------------------------------------------------------
# Switch to gds_role and create projection-ready views
# --------------------------------------------------------------
session.sql("USE WAREHOUSE neo4j_graph_analytics_app_warehouse").collect()

my_user = session.sql("SELECT CURRENT_USER()").collect()[0][0]
session.sql(f"GRANT ROLE gds_role TO USER {my_user}").collect()
session.sql("USE ROLE gds_role").collect()
session.sql("USE DATABASE ga_demo").collect()
session.sql("USE SCHEMA public").collect()

# Node view - just the IDs, which is what graph projections need
session.sql("""
    CREATE OR REPLACE TABLE ga_demo.public.nodes_vw AS
    SELECT machine_id AS nodeId
    FROM ga_demo.public.nodes
""").collect()

# Relationship view - aggregate to ensure one weight per pair
session.sql("""
    CREATE OR REPLACE TABLE ga_demo.public.rels_vw AS
    SELECT
        src_machine_id          AS sourceNodeId,
        dst_machine_id          AS targetNodeId,
        CAST(SUM(throughput_rate) AS FLOAT) AS total_amount
    FROM ga_demo.public.rels
    GROUP BY src_machine_id, dst_machine_id
""").collect()

print("Projection views created. Setup complete - you are ready to start.")

## 1. Thinking in Graphs

Before we run any algorithms, we need a shared vocabulary. Three concepts underpin everything that follows.

### Nodes, relationships and properties

A graph is made of **nodes** (entities) and **relationships** (connections between them). Both can carry **properties**.

In our manufacturing plant:
- Each machine is a **node**. Its `machine_type` and `risk_level` are properties on that node.
- Each material flow connection is a **relationship**. Its `throughput_rate` is a property on that relationship.

We already have this data - it's sitting in the `nodes` and `rels` tables we just created. A graph is not a separate thing you import data into. It is a *lens* on data you already own.

### Why graphs add value for connected data

SQL is excellent at filtering and aggregating. It struggles when the *connections themselves* are what you need to analyze.

Consider the question: **"Which machines would be affected if Machine 3 went offline?"**

In SQL, answering this requires a recursive query - each additional hop through the network multiplies complexity. With five or six hops the query becomes impractical to write and slow to run.

In a graph, following connections is the fundamental operation. Traversing five hops is as natural as traversing one. Graph algorithms are built around this access pattern.

### What a graph projection is

Every algorithm call in Neo4j Graph Analytics includes a `project` block. This tells the app which Snowflake tables to use as nodes and which to use as relationships.

The app reads those tables, builds a temporary in-memory graph structure optimized for traversal, runs the algorithm, writes the results back to a specified Snowflake table and then discards the in-memory structure. **Your data never leaves Snowflake.** The projection is a temporary working structure, not permanent storage.

We'll write our first projection in the next section.


## 2. Your First Graph Projection

Before running any algorithms, let's visualize the graph we've built. This serves two purposes: it confirms that the data loaded correctly and it lets us form an intuition about the structure *before* the algorithms tell us what to think.

Look at the visualization and notice:
- Which machine appears to have the most connections flowing *into* it?
- Are there any machines that seem to sit between otherwise separate parts of the plant?

Hold those observations. The algorithms in Sections 3 and 4 will confirm or challenge them numerically.


In [ ]:
# --------------------------------------------------------------
# Visualize the graph - nodes colored by machine type
# --------------------------------------------------------------
import networkx as nx
import matplotlib.pyplot as plt

nodes_df = session.sql("""
    SELECT machine_id AS nodeId, machine_type
    FROM ga_demo.public.nodes
""").to_pandas()

rels_df = session.sql("""
    SELECT sourceNodeId, targetNodeId
    FROM ga_demo.public.rels_vw
""").to_pandas()

G = nx.DiGraph()
colors_palette = plt.cm.Set3.colors
machine_types  = nodes_df['MACHINE_TYPE'].unique()
color_map      = {mt: colors_palette[i % len(colors_palette)] for i, mt in enumerate(machine_types)}

for _, row in nodes_df.iterrows():
    G.add_node(row['NODEID'], machine_type=row['MACHINE_TYPE'])

for _, row in rels_df.iterrows():
    G.add_edge(row['SOURCENODEID'], row['TARGETNODEID'])

node_colors = [color_map.get(G.nodes[n].get('machine_type', ''), 'grey') for n in G.nodes()]

fig, ax = plt.subplots(figsize=(13, 8))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, ax=ax,
        node_color=node_colors,
        with_labels=True,
        node_size=900,
        edge_color='lightgray',
        font_size=11,
        font_weight='bold',
        alpha=0.9,
        arrows=True,
        arrowsize=12)

legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=color_map[mt], markersize=11, label=mt)
    for mt in machine_types
]
ax.legend(handles=legend_handles, title='Machine Type', loc='best', fontsize=11)
ax.set_title('Manufacturing Plant - Machine Flow Graph', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Connectivity Analysis

### Weakly Connected Components (WCC)

Our first algorithm answers a foundational question: **is this plant one integrated system or does it split into isolated subsystems?**

WCC treats the graph as undirected - it ignores the direction of material flow and asks simply: can every machine reach every other machine through *some* path?

The output assigns each machine a **component ID**. Machines that share a component ID are connected. Multiple component IDs would indicate isolated sub-plants - potentially a blind spot in monitoring or coordination.

### What to expect

Our data was designed to form a single connected system. Let's run the algorithm and verify that.

> **Note on compute pool startup:** If this is your first algorithm call, the Snowpark Container Services compute pool may be in a SUSPENDED state. The first call will take 1-2 minutes while the containers spin up. Subsequent calls will be much faster.


In [ ]:
# --------------------------------------------------------------
# Run WCC
# --------------------------------------------------------------
session.sql("""
    CALL neo4j_graph_analytics.graph.wcc('CPU_X64_XS', {
        'project': {
            'defaultTablePrefix': 'ga_demo.public',
            'nodeTables': ['nodes_vw'],
            'relationshipTables': {
                'rels_vw': {
                    'sourceTable': 'nodes_vw',
                    'targetTable': 'nodes_vw'
                }
            }
        },
        'compute': {},
        'write': [{
            'nodeLabel': 'nodes_vw',
            'outputTable': 'ga_demo.public.nodes_wcc'
        }]
    })
""").collect()

print("WCC complete. Results written to ga_demo.public.nodes_wcc.")

In [ ]:
# --------------------------------------------------------------
# Query WCC results
# --------------------------------------------------------------
wcc_summary = session.sql("""
    SELECT component, COUNT(*) AS machine_count
    FROM ga_demo.public.nodes_wcc
    GROUP BY component
    ORDER BY machine_count DESC
""").to_pandas()

print(f"Number of components found: {len(wcc_summary)}")
print()
print(wcc_summary.to_string(index=False))

## 4. Criticality Analysis

We know the plant is connected. Now: **which machines are most critical?**

We'll use two algorithms that measure criticality in different ways. Running both matters - a machine can be critical for one reason but not the other and the distinction has real operational implications.

### PageRank - flow importance

PageRank asks: which machines receive material from many *well-connected* upstream machines?

A high PageRank score means a machine is a destination for flow from important sources. It's a measure of how much the rest of the plant depends on feeding into this machine. If a high-PageRank machine slows down, it creates a backlog that ripples upstream.

### Betweenness Centrality - structural importance

Betweenness asks a different question: which machines appear most often on the *shortest path between other machines*?

A high Betweenness score means a machine is a structural bridge. It may not handle the most flow, but its position connects otherwise separate parts of the plant. If a high-Betweenness machine goes offline, it disconnects or lengthens paths across the network.

**The key insight:** these two measures are not the same. A machine can score high on one and low on the other. Both matter for maintenance planning.


In [ ]:
# --------------------------------------------------------------
# Run PageRank
# --------------------------------------------------------------
session.sql("""
    CALL neo4j_graph_analytics.graph.page_rank('CPU_X64_XS', {
        'project': {
            'defaultTablePrefix': 'ga_demo.public',
            'nodeTables': ['nodes_vw'],
            'relationshipTables': {
                'rels_vw': {
                    'sourceTable': 'nodes_vw',
                    'targetTable': 'nodes_vw'
                }
            }
        },
        'compute': { 'mutateProperty': 'score' },
        'write': [{
            'nodeLabel': 'nodes_vw',
            'outputTable': 'ga_demo.public.nodes_pagerank',
            'nodeProperty': 'score'
        }]
    })
""").collect()

pr_df = session.sql("""
    SELECT p.nodeid, n.machine_type, n.risk_level, ROUND(p.score, 4) AS pagerank_score
    FROM ga_demo.public.nodes_pagerank p
    JOIN ga_demo.public.nodes n ON p.nodeid = n.machine_id
    ORDER BY pagerank_score DESC
    LIMIT 5
""").to_pandas()

print("Top 5 machines by PageRank score:")
print(pr_df.to_string(index=False))

In [ ]:
# --------------------------------------------------------------
# Visualize PageRank
# Node size proportional to PageRank score, colour by risk level
# --------------------------------------------------------------
import networkx as nx
import matplotlib.pyplot as plt

pr_all = session.sql("SELECT nodeid, score FROM ga_demo.public.nodes_pagerank").to_pandas()
nodes_all = session.sql("SELECT machine_id, risk_level FROM ga_demo.public.nodes").to_pandas()
rels_viz = session.sql("SELECT sourceNodeId, targetNodeId FROM ga_demo.public.rels_vw").to_pandas()

pr_lookup   = dict(zip(pr_all['NODEID'], pr_all['SCORE']))
risk_colors = {'low': 'steelblue', 'medium': 'orange', 'high': 'crimson'}

G2 = nx.DiGraph()
for _, row in nodes_all.iterrows():
    G2.add_node(row['MACHINE_ID'],
                score=pr_lookup.get(row['MACHINE_ID'], 0),
                risk=row['RISK_LEVEL'].lower())
for _, row in rels_viz.iterrows():
    G2.add_edge(row['SOURCENODEID'], row['TARGETNODEID'])

pos2         = nx.spring_layout(G2, seed=42)
node_colors2 = [risk_colors.get(G2.nodes[n]['risk'], 'grey') for n in G2.nodes()]
node_sizes2  = [G2.nodes[n]['score'] * 10000 + 400 for n in G2.nodes()]

fig, ax = plt.subplots(figsize=(13, 8))
nx.draw(G2, pos2, ax=ax,
        node_color=node_colors2, node_size=node_sizes2,
        with_labels=True, edge_color='lightgray',
        font_size=11, font_weight='bold', alpha=0.9,
        arrows=True, arrowsize=12)

legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=c, markersize=12, label=l.capitalize())
    for l, c in risk_colors.items()
]
ax.legend(handles=legend_handles, title='Risk Level', fontsize=11)
ax.set_title('PageRank — Node size reflects influence, colour reflects risk level', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --------------------------------------------------------------
# Run Betweenness Centrality
# --------------------------------------------------------------
session.sql("""
    CALL neo4j_graph_analytics.graph.betweenness('CPU_X64_XS', {
        'project': {
            'defaultTablePrefix': 'ga_demo.public',
            'nodeTables': ['nodes_vw'],
            'relationshipTables': {
                'rels_vw': {
                    'sourceTable': 'nodes_vw',
                    'targetTable': 'nodes_vw'
                }
            }
        },
        'compute': { 'mutateProperty': 'score' },
        'write': [{
            'nodeLabel': 'nodes_vw',
            'outputTable': 'ga_demo.public.nodes_betweenness',
            'nodeProperty': 'score'
        }]
    })
""").collect()

btwn_df = session.sql("""
    SELECT b.nodeid, n.machine_type, n.risk_level, ROUND(b.score, 4) AS betweenness_score
    FROM ga_demo.public.nodes_betweenness b
    JOIN ga_demo.public.nodes n ON b.nodeid = n.machine_id
    ORDER BY betweenness_score DESC
    LIMIT 5
""").to_pandas()

print("Top 5 machines by Betweenness score:")
print(btwn_df.to_string(index=False))

In [ ]:
# --------------------------------------------------------------
# Visualize Betweenness - heatmap colour by score intensity
# --------------------------------------------------------------
import matplotlib.colors as mcolors

btwn_all = session.sql("SELECT nodeid, score FROM ga_demo.public.nodes_betweenness").to_pandas()
btwn_lookup = dict(zip(btwn_all['NODEID'], btwn_all['SCORE']))

G3 = nx.DiGraph()
for _, row in nodes_all.iterrows():
    G3.add_node(row['MACHINE_ID'], score=btwn_lookup.get(row['MACHINE_ID'], 0))
for _, row in rels_viz.iterrows():
    G3.add_edge(row['SOURCENODEID'], row['TARGETNODEID'])

scores3 = [G3.nodes[n]['score'] for n in G3.nodes()]
norm3   = mcolors.Normalize(vmin=min(scores3), vmax=max(scores3))
cmap3   = plt.cm.YlOrRd
colors3 = [cmap3(norm3(s)) for s in scores3]
font_c  = {n: 'white' if norm3(G3.nodes[n]['score']) > 0.5 else 'black' for n in G3.nodes()}

fig, ax = plt.subplots(figsize=(13, 8))
nx.draw(G3, pos2, ax=ax,
        node_color=colors3, node_size=900, with_labels=False,
        edge_color='lightgray', alpha=0.9,
        arrows=True, arrowsize=12)
for node, (x, y) in pos2.items():
    ax.text(x, y, str(node), fontsize=11, fontweight='bold',
            color=font_c[node], ha='center', va='center')

sm = plt.cm.ScalarMappable(cmap=cmap3, norm=norm3)
sm.set_array([])
cb = plt.colorbar(sm, ax=ax, shrink=0.8)
cb.set_label('Betweenness Score', fontsize=12)
ax.set_title('Betweenness Centrality — colour intensity reflects bridge importance', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Structural Similarity

So far we've looked at individual machines - which are most connected, most central, most at risk. This section asks a different question: **which machines play the same structural role in the workflow, even if they are different types?**

This matters for operational planning. Machines with structurally equivalent positions can share maintenance windows, act as backups for each other, or be treated as a unit for risk modeling - even if they look different on paper.

We use two algorithms in sequence:

### Fast Random Projection (FastRP)

FastRP generates a compact embedding vector for each machine by sampling the graph structure around it. Two machines with similar upstream and downstream neighbors will end up with similar embedding vectors, regardless of their type or risk level.

The embedding dimension controls how many values are in each vector. A higher dimension captures more structural detail but takes longer to compute. For this dataset we use **16 dimensions** - a good balance for a 20-node graph.

The raw embedding vectors are not directly interpretable - they are intermediate representations used as input to the next step.

### K-Nearest Neighbor (KNN)

KNN takes the embeddings and finds, for each machine, its most structurally similar peers. Similarity is measured using cosine similarity of the embedding vectors - a score of 1.0 means identical structural position, 0.0 means completely different.

The `topK` parameter controls how many similar peers to find per machine. We set `topK` to **1** to return the single most similar peer for each machine.

> **Note:** KNN operates on node properties rather than graph edges, so its projection block contains no relationship table. This is the one exception to the pattern seen in previous sections.


In [ ]:
# --------------------------------------------------------------
# Run FastRP to generate embeddings
# --------------------------------------------------------------
session.sql("""
    CALL neo4j_graph_analytics.graph.fast_rp('CPU_X64_XS', {
        'project': {
            'defaultTablePrefix': 'ga_demo.public',
            'nodeTables': ['nodes_vw'],
            'relationshipTables': {
                'rels_vw': {
                    'sourceTable': 'nodes_vw',
                    'targetTable': 'nodes_vw'
                }
            }
        },
        'compute': {
            'mutateProperty': 'embedding',
            'embeddingDimension': 16
        },
        'write': [{
            'nodeLabel': 'nodes_vw',
            'outputTable': 'ga_demo.public.nodes_fastrp',
            'nodeProperty': 'embedding'
        }]
    })
""").collect()

import json

emb_preview = session.sql("""
    SELECT nodeid AS machine, embedding
    FROM ga_demo.public.nodes_fastrp
    LIMIT 3
""").to_pandas()

print("FastRP embeddings (16-dimensional vectors per machine):")
for _, row in emb_preview.iterrows():
    values = json.loads(row['EMBEDDING'])
    formatted = [f"{v:+.4f}" for v in values]
    print(f"  Machine {int(row['MACHINE'])}: [{', '.join(formatted)}]")

In [ ]:
# --------------------------------------------------------------
# Run KNN on the embeddings
# Note: no relationship table in this projection - KNN uses
# node embedding properties, not graph edges
# --------------------------------------------------------------
session.sql("""
    CALL neo4j_graph_analytics.graph.knn('CPU_X64_XS', {
        'project': {
            'defaultTablePrefix': 'ga_demo.public',
            'nodeTables': ['nodes_fastrp'],
            'relationshipTables': {}
        },
        'compute': {
            'nodeProperties': ['EMBEDDING'],
            'topK': 1,
            'mutateProperty': 'score',
            'mutateRelationshipType': 'SIMILAR'
        },
        'write': [{
            'outputTable': 'ga_demo.public.nodes_knn',
            'sourceLabel': 'nodes_fastrp',
            'targetLabel': 'nodes_fastrp',
            'relationshipType': 'SIMILAR',
            'relationshipProperty': 'score'
        }]
    })
""").collect()

knn_df = session.sql("""
    SELECT
        k.sourcenodeid  AS machine,
        n_src.machine_type AS machine_type,
        k.targetnodeid  AS similar_to,
        n_tgt.machine_type AS similar_type,
        ROUND(k.score, 4)  AS similarity
    FROM ga_demo.public.nodes_knn k
    JOIN ga_demo.public.nodes n_src ON k.sourcenodeid = n_src.machine_id
    JOIN ga_demo.public.nodes n_tgt ON k.targetnodeid = n_tgt.machine_id
    ORDER BY similarity DESC
    LIMIT 8
""").to_pandas()

print("Top KNN pairs by structural similarity:")
print(knn_df.to_string(index=False))

In [ ]:
# --------------------------------------------------------------
# Visualize KNN results as a similarity matrix
# --------------------------------------------------------------
import numpy as np

knn_all   = session.sql("SELECT sourcenodeid, targetnodeid, score FROM ga_demo.public.nodes_knn").to_pandas()
nodes_ids = sorted(nodes_all['MACHINE_ID'].unique())
n         = len(nodes_ids)
idx       = {mid: i for i, mid in enumerate(nodes_ids)}

matrix = np.zeros((n, n))
for _, row in knn_all.iterrows():
    i = idx.get(row['SOURCENODEID'])
    j = idx.get(row['TARGETNODEID'])
    if i is not None and j is not None:
        matrix[i][j] = row['SCORE']
        matrix[j][i] = row['SCORE']

fig, ax = plt.subplots(figsize=(13, 9))
im = ax.imshow(matrix, cmap='Greens', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(nodes_ids, fontsize=11)
ax.set_yticklabels(nodes_ids, fontsize=11)

for i in range(n):
    for j in range(n):
        if matrix[i][j] > 0:
            ax.text(j, i, f'{matrix[i][j]:.2f}',
                    ha='center', va='center', fontsize=10, fontweight='bold',
                    color='white' if matrix[i][j] > 0.5 else 'black')

cb = plt.colorbar(im, ax=ax, shrink=0.8)
cb.set_label('Similarity Score', fontsize=12)
ax.set_title('KNN Structural Similarity - darker cells indicate more similar machines', fontsize=14)
ax.set_xticks(np.arange(-0.5, n, 1), minor=True)
ax.set_yticks(np.arange(-0.5, n, 1), minor=True)
ax.grid(which='minor', color='black', linewidth=1.2)
ax.tick_params(which='minor', size=0)
plt.tight_layout()
plt.show()

## Sections Complete

We've run four graph algorithms on the manufacturing plant dataset. Here's what each one answered:

| Algorithm | Question answered | Top result |
|---|---|---|
| WCC | Is the plant one connected system? | Yes -- single component |
| PageRank | Which machine receives the most flow from important sources? | Machine 20 |
| Betweenness | Which machine is the most critical structural bridge? | Machine 3 |
| FastRP + KNN | Which machines play equivalent structural roles? | See your output |

Continue to Sections 6 and 7 to take the analysis further -- simulating a machine failure and detecting operational communities within the plant.


## 6. Failure Simulation

Static risk analysis tells us which machines are currently important. This section turns that into a dynamic tool: **what actually happens to the rest of the plant when Machine 3 goes offline?**

We simulate the failure by creating filtered views that exclude the chosen machine and all its connections. We then re-run PageRank and Betweenness on the degraded graph and compare normalized scores before and after.

Normalization matters here: raw scores shrink after failure because the graph is smaller. We divide each score by the sum of all scores in that run so we're comparing *relative importance within each graph*, not absolute values.

**What to watch for:** machines that were not flagged as high risk in the baseline analysis but gain significant Betweenness importance after the failure. These are hidden risks that static analysis alone would miss.

**Experimenting:** change the value for `EXCLUDED` in the next cell to simulate failure of any machine in the plant and re-run the section to see how the network responds.


In [ ]:
# --------------------------------------------------------------
# Select the machine to remove from the graph
# --------------------------------------------------------------
# Change this value to simulate failure of a different machine.
EXCLUDED = 3

In [ ]:
# --------------------------------------------------------------
# Create failure views
# --------------------------------------------------------------
session.sql(f"""
    CREATE OR REPLACE VIEW ga_demo.public.nodes_failure_vw AS
    SELECT machine_id AS nodeId
    FROM ga_demo.public.nodes
    WHERE machine_id != {EXCLUDED}
""").collect()

session.sql(f"""
    CREATE OR REPLACE VIEW ga_demo.public.rels_failure_vw AS
    SELECT
        src_machine_id AS sourceNodeId,
        dst_machine_id AS targetNodeId,
        CAST(SUM(throughput_rate) AS FLOAT) AS total_amount
    FROM ga_demo.public.rels
    WHERE src_machine_id != {EXCLUDED}
      AND dst_machine_id != {EXCLUDED}
    GROUP BY src_machine_id, dst_machine_id
""").collect()

print(f"Failure views created. Machine {EXCLUDED} has been removed from the graph.")

In [ ]:
# --------------------------------------------------------------
# Re-run PageRank on the degraded graph
# --------------------------------------------------------------
session.sql("""
    CALL neo4j_graph_analytics.graph.page_rank('CPU_X64_XS', {
        'project': {
            'defaultTablePrefix': 'ga_demo.public',
            'nodeTables': ['nodes_failure_vw'],
            'relationshipTables': {
                'rels_failure_vw': {
                    'sourceTable': 'nodes_failure_vw',
                    'targetTable': 'nodes_failure_vw'
                }
            }
        },
        'compute': { 'mutateProperty': 'score' },
        'write': [{
            'nodeLabel': 'nodes_failure_vw',
            'outputTable': 'ga_demo.public.nodes_failure_pagerank',
            'nodeProperty': 'score'
        }]
    })
""").collect()

pr_compare = session.sql("""
    SELECT
        b.nodeid,
        n.machine_type,
        n.risk_level,
        ROUND(b.score / SUM(b.score) OVER (), 4) AS baseline_pct,
        ROUND(f.score / SUM(f.score) OVER (), 4) AS failure_pct,
        ROUND((f.score / SUM(f.score) OVER ()) - (b.score / SUM(b.score) OVER ()), 4) AS delta
    FROM ga_demo.public.nodes_pagerank b
    JOIN ga_demo.public.nodes_failure_pagerank f ON b.nodeid = f.nodeid
    JOIN ga_demo.public.nodes n ON b.nodeid = n.machine_id
    ORDER BY delta DESC
    LIMIT 8
""").to_pandas()

print("PageRank delta after failure (top gainers):")
print(pr_compare.to_string(index=False))

In [ ]:
# --------------------------------------------------------------
# Re-run Betweenness on the degraded graph
# --------------------------------------------------------------
session.sql("""
    CALL neo4j_graph_analytics.graph.betweenness('CPU_X64_XS', {
        'project': {
            'defaultTablePrefix': 'ga_demo.public',
            'nodeTables': ['nodes_failure_vw'],
            'relationshipTables': {
                'rels_failure_vw': {
                    'sourceTable': 'nodes_failure_vw',
                    'targetTable': 'nodes_failure_vw'
                }
            }
        },
        'compute': { 'mutateProperty': 'score' },
        'write': [{
            'nodeLabel': 'nodes_failure_vw',
            'outputTable': 'ga_demo.public.nodes_failure_btwn',
            'nodeProperty': 'score'
        }]
    })
""").collect()

btwn_compare = session.sql("""
    SELECT
        b.nodeid,
        n.machine_type,
        n.risk_level,
        ROUND(b.score / NULLIF(SUM(b.score) OVER (), 0), 4) AS baseline_pct,
        ROUND(f.score / NULLIF(SUM(f.score) OVER (), 0), 4) AS failure_pct,
        ROUND((f.score / NULLIF(SUM(f.score) OVER (), 0))
            - (b.score / NULLIF(SUM(b.score) OVER (), 0)), 4) AS delta
    FROM ga_demo.public.nodes_betweenness b
    JOIN ga_demo.public.nodes_failure_btwn f ON b.nodeid = f.nodeid
    JOIN ga_demo.public.nodes n ON b.nodeid = n.machine_id
    ORDER BY delta DESC
    LIMIT 8
""").to_pandas()

print("Betweenness delta after failure (top gainers):")
print(btwn_compare.to_string(index=False))

In [ ]:
# --------------------------------------------------------------
# Bar chart - Betweenness delta by machine
# --------------------------------------------------------------
import pandas as pd

baseline_b = session.sql("SELECT nodeid, score FROM ga_demo.public.nodes_betweenness").to_pandas()
failure_b  = session.sql("SELECT nodeid, score FROM ga_demo.public.nodes_failure_btwn").to_pandas()
nodes_meta = session.sql("SELECT machine_id, machine_type, risk_level FROM ga_demo.public.nodes").to_pandas()

merged = baseline_b.merge(failure_b, on='NODEID', suffixes=('_base', '_fail'))
merged = merged.merge(nodes_meta, left_on='NODEID', right_on='MACHINE_ID')
merged['DELTA'] = (
    (merged['SCORE_fail'] / merged['SCORE_fail'].sum()) -
    (merged['SCORE_base'] / merged['SCORE_base'].sum())
)
merged = merged.sort_values('DELTA', ascending=False).head(10)

bar_colors = merged['RISK_LEVEL'].str.lower().map(risk_colors).fillna('grey')

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(merged['NODEID'].astype(str), merged['DELTA'], color=bar_colors)
ax.set_xlabel('Machine ID', fontsize=13)
ax.set_ylabel('Betweenness Delta (normalized)', fontsize=13)
ax.set_title(f'Change in Betweenness Centrality after Machine {EXCLUDED} failure', fontsize=14)
ax.tick_params(axis='both', labelsize=12)

legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=c, label=l.capitalize())
    for l, c in risk_colors.items()
]
ax.legend(handles=legend_handles, title='Risk Level', fontsize=11)
plt.tight_layout()
plt.show()

print()
print("Machines with the largest positive delta have absorbed the most structural")
print("importance following the failure. Pay attention to machines that combine a")
print("large delta with a high risk level - these are emergent failure points.")

## 7. Community Detection

The previous sections analyzed individual machines. This section asks: **does the plant naturally organize itself into clusters?**

### Louvain Community Detection

Louvain finds groups of machines that are more densely connected to each other than to the rest of the network. These communities often correspond to real operational sub-units - parallel production lines, shared workflow stages, or tightly coupled machine groups.

After running Louvain, we join the results back to our criticality scores to see which community carries the most concentrated risk - and whether the community structure helps explain why the failure simulation produced the results it did.


In [ ]:
# --------------------------------------------------------------
# Run Louvain
# --------------------------------------------------------------
session.sql("""
    CALL neo4j_graph_analytics.graph.louvain('CPU_X64_XS', {
        'project': {
            'defaultTablePrefix': 'ga_demo.public',
            'nodeTables': ['nodes_vw'],
            'relationshipTables': {
                'rels_vw': {
                    'sourceTable': 'nodes_vw',
                    'targetTable': 'nodes_vw'
                }
            }
        },
        'compute': { 'mutateProperty': 'community' },
        'write': [{
            'nodeLabel': 'nodes_vw',
            'outputTable': 'ga_demo.public.nodes_louvain',
            'nodeProperty': 'community'
        }]
    })
""").collect()

community_summary = session.sql("""
    SELECT
        l.community,
        COUNT(*) AS machine_count,
        LISTAGG(n.machine_id, ', ') WITHIN GROUP (ORDER BY n.machine_id) AS machines,
        SUM(CASE WHEN n.risk_level = 'high'   THEN 1 ELSE 0 END) AS high_risk,
        SUM(CASE WHEN n.risk_level = 'medium' THEN 1 ELSE 0 END) AS medium_risk,
        SUM(CASE WHEN n.risk_level = 'low'    THEN 1 ELSE 0 END) AS low_risk
    FROM ga_demo.public.nodes_louvain l
    JOIN ga_demo.public.nodes n ON l.nodeid = n.machine_id
    GROUP BY l.community
    ORDER BY machine_count DESC
""").to_pandas()

print("Community summary:")
print(community_summary.to_string(index=False))

In [ ]:
# --------------------------------------------------------------
# Visualize communities
# Compare this with the Section 2 type-coloured graph to see
# whether communities align with machine types or cut across them
# --------------------------------------------------------------
import matplotlib.cm as cm

louvain_df = session.sql("SELECT nodeid, community FROM ga_demo.public.nodes_louvain").to_pandas()

communities = sorted(louvain_df['COMMUNITY'].unique())
palette     = cm.Set2.colors
comm_colors = {c: palette[i % len(palette)] for i, c in enumerate(communities)}

G4 = nx.DiGraph()
for _, row in louvain_df.iterrows():
    G4.add_node(row['NODEID'], community=row['COMMUNITY'])
for _, row in rels_viz.iterrows():
    G4.add_edge(row['SOURCENODEID'], row['TARGETNODEID'])

node_colors4 = [comm_colors[G4.nodes[n]['community']] for n in G4.nodes()]

fig, ax = plt.subplots(figsize=(13, 8))
nx.draw(G4, pos2, ax=ax,
        node_color=node_colors4, with_labels=True, node_size=900,
        edge_color='lightgray', font_size=11, font_weight='bold',
        alpha=0.9, arrows=True, arrowsize=12)

legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=comm_colors[c], markersize=12,
               label=f'Community {c}')
    for c in communities
]
ax.legend(handles=legend_handles, title='Community', loc='best', fontsize=11)
ax.set_title('Louvain Community Detection — colour by community membership', fontsize=14)
plt.tight_layout()
plt.show()

print()
print("Notice where Machine 3 sits. Its community membership explains")
print("why its failure had such a large effect on the secondary production line.")

## 8. Risk Summary

We've run seven sections of analysis. This final cell brings it all together into a single risk summary table joining all four algorithm outputs.


In [ ]:
# --------------------------------------------------------------
# Risk summary
# --------------------------------------------------------------
risk_summary = session.sql("""
    SELECT
        n.machine_id,
        n.machine_type,
        n.risk_level,
        ROUND(p.score, 4) AS pagerank_score,
        ROUND(b.score, 4) AS betweenness_score,
        l.community
    FROM ga_demo.public.nodes n
    JOIN ga_demo.public.nodes_pagerank p    ON n.machine_id = p.nodeid
    JOIN ga_demo.public.nodes_betweenness b ON n.machine_id = b.nodeid
    JOIN ga_demo.public.nodes_louvain l     ON n.machine_id = l.nodeid
    ORDER BY pagerank_score DESC
""").to_pandas()

print("Risk summary -- all 20 machines:")
print(risk_summary.to_string(index=False))